In [1]:
!pip install git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git

  Cloning https://github.com/MahmoudAshraf97/ctc-forced-aligner.git to /tmp/pip-req-build-3bcg5ot2
  Running command git clone --filter=blob:none --quiet https://github.com/MahmoudAshraf97/ctc-forced-aligner.git /tmp/pip-req-build-3bcg5ot2
  Resolved https://github.com/MahmoudAshraf97/ctc-forced-aligner.git to commit 1f0a5f860d3d9daf3d94edb1c7d18f90d1702e5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.7/930.7 kB 42.4 MB/s eta 0:00:00
  Created wheel for ctc-forced-aligner: filename=ctc_forced_aligner-0.3.0-cp312-cp312-linux_x86_64.whl size=96942 sha256=c4caccf3928731f7967a222a0e1c85f68046a3cc0a930e2769c87927add7e67d
  Stored in directory: /tmp/pip-ephem-wheel-cache-dzm3xqen/wheels/3c/d4/13/bf2c0f2e6af318a8f505400dfb51fa789db37aab64222656dd
Successfully built ctc-forced-a

In [7]:
!pip install resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.8 MB/s eta 0:00:00


In [9]:
import torch
import soundfile as sf
import numpy as np
import json
from ctc_forced_aligner import (
    load_alignment_model,
    generate_emissions,
    preprocess_text,
    get_alignments,
    get_spans,
    postprocess_results,
)

# Load audio manually using soundfile; converts to mono and 16kHz if needed
def load_audio_manual(audio_path, dtype, device):
    audio_np, sr = sf.read(audio_path)
    if len(audio_np.shape) > 1 and audio_np.shape[1] > 1:
        audio_np = np.mean(audio_np, axis=1)
    if sr != 16000:
        import resampy
        audio_np = resampy.resample(audio_np, sr, 16000)
    audio_tensor = torch.from_numpy(audio_np).to(dtype).to(device)
    return audio_tensor

# Paths to your audio and transcript files
audio_path = "/content/audio_20250821085054.wav"
text_path = "/content/transcript.txt"

# Device and batch config
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16

# Load pretrained alignment model (default MMS aligned for Arabic)
alignment_model, alignment_tokenizer = load_alignment_model(
    device,
    dtype=torch.float16 if device == "cuda" else torch.float32,
)

# Load audio tensor
audio_waveform = load_audio_manual(audio_path, alignment_model.dtype, alignment_model.device)

# Read transcript text, strip newlines
with open(text_path, "r", encoding="utf-8") as f:
    text = f.read().replace("\n", " ").strip()

# Preprocess text with romanization enabled to fix tokenization mismatch error
tokens_starred, text_starred = preprocess_text(
    text,
    romanize=True,    # Enable romanization (critical for Arabic script)
    language="ara",   # ISO 639-3 Arabic language code
)

# Generate acoustic features (emissions) from audio
emissions, stride = generate_emissions(
    alignment_model, audio_waveform, batch_size=batch_size
)

# Get token alignments from emissions and tokenized transcript
segments, scores, blank_token = get_alignments(
    emissions,
    tokens_starred,
    alignment_tokenizer,
)

# Extract time spans for each token, assert token match fixed because of romanize=True
spans = get_spans(tokens_starred, segments, blank_token)

# Postprocess timestamps into human-readable word-level word timings
word_timestamps = postprocess_results(text_starred, spans, stride, scores)

# Save word-level timestamps to JSON output
output_data = {
    "audio_file": audio_path,
    "language": "ara",
    "word_timestamps": word_timestamps,
}

with open("precise_alignment.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(word_timestamps)} word-level timestamps in precise_alignment.json")


Saved 452 word-level timestamps in precise_alignment.json


KeyError: 'word'

In [14]:
import os
import json
import soundfile as sf

# Input files
audio_path = "/content/audio_20250821085054.wav"
transcript_path = "/content/transcript.txt"
alignment_json_path = "/content/precise_alignment.json"

# Output folder
output_dir = "/content/split_output"
os.makedirs(output_dir, exist_ok=True)

# Robust transcript loader with fallback encodings
def load_transcript(path):
    encodings_to_try = ['utf-8', 'cp1256', 'iso-8859-6', 'latin1']
    for enc in encodings_to_try:
        try:
            with open(path, 'r', encoding=enc) as f:
                lines = [line.strip() for line in f if line.strip()]
            print(f"Loaded transcript with encoding: {enc}")
            return lines
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Failed to decode transcript with tried encodings: {encodings_to_try}")

# Load sentences from transcript file
sentences = load_transcript(transcript_path)

# Load precise alignment timestamps
with open(alignment_json_path, 'r', encoding='utf-8') as f:
    alignment_data = json.load(f)
word_timestamps = alignment_data["word_timestamps"]

# Group word timestamps by sentences, matching words ignoring spaces/punctuations
def group_word_timestamps_by_sentence(sentences, word_timestamps):
    grouped = []
    word_idx = 0
    num_words = len(word_timestamps)

    for sent in sentences:
        sent_words = []
        # Normalize sentence: remove spaces and common arabic punctuation for matching
        sent_normalized = sent.replace(" ", "").replace(".", "").replace("،", "").replace("!", "")
        concat_words = ""
        start_time = None
        end_time = None
        while word_idx < num_words and len(concat_words) < len(sent_normalized):
            word_txt = word_timestamps[word_idx]["text"].replace(" ", "").replace(".", "").replace("،", "").replace("!", "")
            concat_words += word_txt
            if start_time is None:
                start_time = word_timestamps[word_idx]["start"]
            end_time = word_timestamps[word_idx]["end"]
            sent_words.append(word_timestamps[word_idx])
            word_idx += 1
        grouped.append({
            "sentence": sent,
            "start": start_time,
            "end": end_time,
            "words": sent_words,
        })
    return grouped

sentence_spans = group_word_timestamps_by_sentence(sentences, word_timestamps)

# Load full audio
audio, sr = sf.read(audio_path)

# Split and save audio/transcripts per sentence
for i, s in enumerate(sentence_spans):
    start_sample = int(s["start"] * sr)
    end_sample = int(s["end"] * sr)
    # Boundaries check
    start_sample = max(0, start_sample)
    end_sample = min(len(audio), end_sample)
    audio_chunk = audio[start_sample:end_sample]

    # Filenames with zero-padded index
    out_audio_path = os.path.join(output_dir, f"audio_{i:02d}.wav")
    out_text_path = os.path.join(output_dir, f"transcript_{i:02d}.txt")

    # Write audio chunk
    sf.write(out_audio_path, audio_chunk, sr)
    # Write transcript sentence
    with open(out_text_path, "w", encoding="utf-8") as ftext:
        ftext.write(s["sentence"])

    print(f"Saved chunk {i:02d} - audio: {out_audio_path}, transcript: {out_text_path}")

print(f"\nAll {len(sentence_spans)} chunks saved in folder: {output_dir}")

Loaded transcript with encoding: utf-8
Saved chunk 00 - audio: /content/split_output/audio_00.wav, transcript: /content/split_output/transcript_00.txt
Saved chunk 01 - audio: /content/split_output/audio_01.wav, transcript: /content/split_output/transcript_01.txt
Saved chunk 02 - audio: /content/split_output/audio_02.wav, transcript: /content/split_output/transcript_02.txt
Saved chunk 03 - audio: /content/split_output/audio_03.wav, transcript: /content/split_output/transcript_03.txt
Saved chunk 04 - audio: /content/split_output/audio_04.wav, transcript: /content/split_output/transcript_04.txt
Saved chunk 05 - audio: /content/split_output/audio_05.wav, transcript: /content/split_output/transcript_05.txt
Saved chunk 06 - audio: /content/split_output/audio_06.wav, transcript: /content/split_output/transcript_06.txt
Saved chunk 07 - audio: /content/split_output/audio_07.wav, transcript: /content/split_output/transcript_07.txt
Saved chunk 08 - audio: /content/split_output/audio_08.wav, trans